# 📚 Hybrid Book Recommendation System — GoodBooks-10k Edition
**Content-Based Filtering + Adaptive SVD Collaborative Filtering + Meta-Regression**

| Aspect | Details |
|--------|---------|
| Dataset | GoodBooks-10k — 10,000 books, ~6 million **real** user ratings |
| Content model | `all-MiniLM-L6-v2` sentence embeddings + FAISS IndexFlatIP |
| Collaborative model | SVD (Surprise) trained on real held-out ratings |
| Hybrid fusion | Adaptive user-weight blend + LinearRegression calibrator |
| Cold-start fallback | Open Library Search API for books outside GoodBooks |
| Covers | Open Library Covers API (by ISBN) |
| Evaluation | RMSE · Precision@K · Recall@K · NDCG@K on real test split |
| Ablation | Content-only vs CF-only vs Hybrid comparison |

## 1. Install Dependencies

In [1]:
# !pip install numpy==1.26.4

In [2]:
!pip install -q sentence-transformers faiss-cpu scikit-learn scikit-surprise ipywidgets requests pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 88.2 MB/s eta 0:00:00


## 2. Import Libraries

In [3]:
import requests, time, os
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
import faiss

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_split
from surprise import accuracy

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print('✅ All libraries imported successfully!')

✅ All libraries imported successfully!


## 3. Download GoodBooks-10k Dataset

We pull the dataset directly from the official GitHub repo.
- `books.csv` — metadata for 10,000 books (title, authors, ISBN, ratings)
- `ratings.csv` — ~6 million explicit 1–5 star ratings from real users

No Kaggle account or API key needed.

In [4]:
DATA_DIR = 'goodbooks_data'
os.makedirs(DATA_DIR, exist_ok=True)

FILES = {
    'books.csv':   'https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv',
    'ratings.csv': 'https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv',
}

for fname, url in FILES.items():
    dest = os.path.join(DATA_DIR, fname)
    if os.path.exists(dest):
        print(f'  {fname} already downloaded — skipping.')
        continue
    print(f'  Downloading {fname} …', end=' ', flush=True)
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with open(dest, 'wb') as f:
        f.write(r.content)
    print(f'done ({len(r.content)//1024} KB)')

print('\n✅ Dataset ready.')


✅ Dataset ready.


## 4. Load & Clean the Dataset

We retain the columns needed for embedding and recommendation, drop rows with missing titles,
and build a `text_feature` string (`title + authors`) used for sentence embedding.
A contiguous `book_idx` (0-based) is created so FAISS and SVD indices align.

In [5]:
books_raw = pd.read_csv(os.path.join(DATA_DIR, 'books.csv'))

KEEP = ['book_id','title','authors','average_rating','ratings_count',
        'isbn','isbn13','original_publication_year','image_url']
books = books_raw[KEEP].copy().dropna(subset=['title']).reset_index(drop=True)

# Popularity score (MinMax on ratings_count)
scaler = MinMaxScaler()
books['popularity_score'] = scaler.fit_transform(
    books['ratings_count'].fillna(0).values.reshape(-1, 1)
)

# Text feature for embedding
books['text_feature'] = books['title'].fillna('') + ' ' + books['authors'].fillna('')

# Load real ratings
ratings_raw = pd.read_csv(os.path.join(DATA_DIR, 'ratings.csv'))
valid_ids   = set(books['book_id'].values)
ratings_df  = ratings_raw[ratings_raw['book_id'].isin(valid_ids)].copy()

# Map book_id → 0-based book_idx (matches DataFrame index == FAISS index)
id_to_idx = {bid: idx for idx, bid in enumerate(books['book_id'])}
ratings_df['book_idx'] = ratings_df['book_id'].map(id_to_idx).astype(int)

print(f'Books     : {len(books):,}')
print(f'Ratings   : {len(ratings_df):,}  '
      f'(users: {ratings_df["user_id"].nunique():,}, '
      f'books: {ratings_df["book_id"].nunique():,})')
print(f'Avg rating: {ratings_df["rating"].mean():.2f}')
books[['title','authors','average_rating','ratings_count']].head(5)

Books     : 10,000
Ratings   : 5,976,479  (users: 53,424, books: 10,000)
Avg rating: 3.92


,title,authors,average_rating,ratings_count
0,"The Hunger Games (The Hunger Games, #1)",Suzanne Collins,4.34,4780653
1,Harry Potter and the Sorcerer's Stone (Harry P...,"J.K. Rowling, Mary GrandPré",4.44,4602479
2,"Twilight (Twilight, #1)",Stephenie Meyer,3.57,3866839
3,To Kill a Mockingbird,Harper Lee,4.25,3198671
4,The Great Gatsby,F. Scott Fitzgerald,3.89,2683664


## 5. Content-Based Filtering (Sentence Embeddings + FAISS)

Each book's `text_feature` is encoded with `all-MiniLM-L6-v2` (384-dim).
Vectors are L2-normalised so inner-product search gives cosine similarity.
`IndexFlatIP` performs exact search — no approximation.

In [6]:
print('Encoding book embeddings (~2 min on CPU) …')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
texts = books['text_feature'].fillna('').tolist()
embeddings = embed_model.encode(
    texts, show_progress_bar=True, batch_size=128
).astype('float32')

faiss.normalize_L2(embeddings)
dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

print(f'\n✅ FAISS index: {faiss_index.ntotal:,} vectors  |  dim = {dim}')

Encoding book embeddings (~2 min on CPU) …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]


✅ FAISS index: 10,000 vectors  |  dim = 384


In [7]:
def encode_query(text):
    vec = embed_model.encode([text]).astype('float32')
    faiss.normalize_L2(vec)
    return vec


def content_recommend_from_vec(query_vec, top_n=5, exclude_idx=None):
    k = top_n + (1 if exclude_idx is not None else 0) + 10
    scores, indices = faiss_index.search(query_vec, k)
    results = []
    for i, s in zip(indices[0], scores[0]):
        if i == exclude_idx:
            continue
        results.append((int(i), float(s)))
        if len(results) >= top_n:
            break
    return results


# Smoke test
q = encode_query('epic fantasy dragons magic quest')
print('Content-based test — "epic fantasy dragons magic quest":')
for idx, s in content_recommend_from_vec(q, top_n=5):
    row = books.iloc[idx]
    print(f'  [{s:.3f}] {row["title"]} — {row["authors"]}')

Content-based test — "epic fantasy dragons magic quest":
  [0.548] Searching for Dragons (Enchanted Forest Chronicles, #2) — Patricia C. Wrede, Peter de Sève
  [0.544] Dragons of Winter Night (Dragonlance: Chronicles, #2) — Margaret Weis, Tracy Hickman
  [0.542] Dealing with Dragons (Enchanted Forest Chronicles, #1) — Patricia C. Wrede, Peter de Sève
  [0.535] Dragons of Spring Dawning (Dragonlance: Chronicles, #3) — Margaret Weis, Tracy Hickman
  [0.531] Dragons of Autumn Twilight  (Dragonlance: Chronicles, #1) — Margaret Weis, Tracy Hickman


## 6. Collaborative Filtering — SVD on Real GoodBooks Ratings

We train Surprise's SVD on the **actual user ratings** (no synthetic data).
An 80/20 stratified split is used; the test set is held out and never seen during training.

A 500 k-row sample is used by default for faster training in demo settings.
Set `SAMPLE = None` to train on all ~6 M ratings (slower but higher quality).

In [8]:
SAMPLE = 500_000   # set to None to use all ~6M ratings

if SAMPLE and len(ratings_df) > SAMPLE:
    ratings_sample = ratings_df.sample(SAMPLE, random_state=42)
    print(f'Using {SAMPLE:,}-row sample for SVD training.')
else:
    ratings_sample = ratings_df
    print(f'Using all {len(ratings_df):,} ratings.')

reader        = Reader(rating_scale=(1, 5))
surprise_data = Dataset.load_from_df(
    ratings_sample[['user_id', 'book_idx', 'rating']], reader
)
trainset, testset = surprise_split(surprise_data, test_size=0.20, random_state=42)

svd_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd_model.fit(trainset)

# Track user activity across the FULL ratings set
user_activity = ratings_df.groupby('user_id').size().to_dict()

print('\n✅ SVD model trained on real GoodBooks-10k ratings!')

Using 500,000-row sample for SVD training.

✅ SVD model trained on real GoodBooks-10k ratings!


## 7. Evaluation — RMSE · Precision@K · Recall@K · NDCG@K

All metrics are computed on the **held-out test split** of real ratings the model never saw.

In [9]:
svd_predictions = svd_model.test(testset)
rmse = accuracy.rmse(svd_predictions, verbose=False)


def precision_recall_at_k(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))
    prec, rec = {}, {}
    for uid, ur in user_est_true.items():
        ur.sort(key=lambda x: x[0], reverse=True)
        n_rel       = sum(t >= threshold for _, t in ur)
        n_rec_k     = sum(e >= threshold for e, _ in ur[:k])
        n_rel_rec_k = sum((t >= threshold) and (e >= threshold) for e, t in ur[:k])
        prec[uid]   = n_rel_rec_k / n_rec_k if n_rec_k else 0
        rec[uid]    = n_rel_rec_k / n_rel   if n_rel   else 0
    return prec, rec


def ndcg_at_k(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))
    ndcg_scores = []
    for uid, ur in user_est_true.items():
        ur.sort(key=lambda x: x[0], reverse=True)
        dcg   = sum((1 if t >= threshold else 0) / np.log2(r + 2)
                    for r, (e, t) in enumerate(ur[:k]))
        ideal = sorted(ur, key=lambda x: x[1], reverse=True)[:k]
        idcg  = sum((1 if t >= threshold else 0) / np.log2(r + 2)
                    for r, (e, t) in enumerate(ideal))
        ndcg_scores.append(dcg / idcg if idcg else 0)
    return float(np.mean(ndcg_scores))


precisions, recalls = precision_recall_at_k(svd_predictions)
ndcg                = ndcg_at_k(svd_predictions)
avg_precision       = float(np.mean(list(precisions.values())))
avg_recall          = float(np.mean(list(recalls.values())))

print('\n========== Evaluation Results (real held-out test split) ==========')
print(f'  RMSE          : {rmse:.4f}')
print(f'  Precision@10  : {avg_precision:.4f}')
print(f'  Recall@10     : {avg_recall:.4f}')
print(f'  NDCG@10       : {ndcg:.4f}')
print('==================================================================')


========== Evaluation Results (real held-out test split) ==========
  RMSE          : 0.9160
  Precision@10  : 0.6768
  Recall@10     : 0.7586
  NDCG@10       : 0.7958


## 8. Ablation Study — Content-Only vs CF-Only vs Hybrid

We evaluate each component in isolation on the same held-out test split,
then compare against the full hybrid to confirm that combining signals improves ranking quality.

In [10]:
def ranking_metrics(predictions_list, k=10, threshold=3.5):
    """Compute Precision@K, Recall@K, NDCG@K from a list of (score, true_r) per user."""
    user_data = defaultdict(list)
    for uid, score, true_r in predictions_list:
        user_data[uid].append((score, true_r))
    prec_list, rec_list, ndcg_list = [], [], []
    for uid, ur in user_data.items():
        ur.sort(key=lambda x: x[0], reverse=True)
        n_rel       = sum(t >= threshold for _, t in ur)
        n_rec_k     = sum(e >= threshold for e, _ in ur[:k])
        n_rel_rec_k = sum((t >= threshold) and (e >= threshold) for e, t in ur[:k])
        prec_list.append(n_rel_rec_k / n_rec_k if n_rec_k else 0)
        rec_list.append(n_rel_rec_k / n_rel   if n_rel   else 0)
        dcg   = sum((1 if t >= threshold else 0) / np.log2(r + 2)
                    for r, (e, t) in enumerate(ur[:k]))
        ideal = sorted(ur, key=lambda x: x[1], reverse=True)[:k]
        idcg  = sum((1 if t >= threshold else 0) / np.log2(r + 2)
                    for r, (e, t) in enumerate(ideal))
        ndcg_list.append(dcg / idcg if idcg else 0)
    return {
        'Precision@10': round(float(np.mean(prec_list)), 4),
        'Recall@10':    round(float(np.mean(rec_list)), 4),
        'NDCG@10':      round(float(np.mean(ndcg_list)), 4),
    }


# Build scored lists for each mode
cf_list, content_list, hybrid_list = [], [], []
for uid, iid, true_r, est, _ in svd_predictions:
    idx      = int(iid)
    pop      = float(books.iloc[idx]['popularity_score']) if 0 <= idx < len(books) else 0
    cos_sim  = float(embeddings[idx] @ embeddings[idx]) if 0 <= idx < len(embeddings) else 0
    cf_score = est
    content_score = cos_sim * 5   # scale to ~0-5 range
    hybrid_score  = 0.7 * cf_score + 0.3 * content_score
    cf_list.append((uid, cf_score, true_r))
    content_list.append((uid, content_score, true_r))
    hybrid_list.append((uid, hybrid_score, true_r))

ablation_df = pd.DataFrame(
    [ranking_metrics(content_list),
     ranking_metrics(cf_list),
     ranking_metrics(hybrid_list)],
    index=['Content-Only', 'CF-Only (SVD)', 'Hybrid (70% CF + 30% Content)']
)
print('\n========== Ablation Study (K=10, real test split) ==========')
print(ablation_df.to_string())
print('============================================================')


========== Ablation Study (K=10, real test split) ==========
                               Precision@10  Recall@10  NDCG@10
Content-Only                         0.6935     0.8410   0.7817
CF-Only (SVD)                        0.6768     0.7586   0.7958
Hybrid (70% CF + 30% Content)        0.6936     0.8382   0.7958


## 9. Open Library Cover & Cold-Start Helper

Cover images are fetched from the Open Library Covers API using the book's ISBN.
For books outside the GoodBooks catalogue, the Open Library Search API provides a
real-time fallback so recommendations are never impossible.

In [11]:
OL_COV_BASE = 'https://covers.openlibrary.org'
OL_BASE     = 'https://openlibrary.org'
OL_HEADERS  = {'User-Agent': 'HybridBookRecommender-BTech (student@example.com)'}
_CACHE      = {}


def cover_url_by_isbn(isbn, size='M'):
    if not isbn or str(isbn) in ('nan', ''):
        return None
    return f'{OL_COV_BASE}/b/isbn/{isbn}-{size}.jpg'


def _ol_get(url, params=None, delay=0.35):
    key = url + str(sorted((params or {}).items()))
    if key in _CACHE:
        return _CACHE[key]
    time.sleep(delay)
    try:
        r = requests.get(url, params=params, headers=OL_HEADERS, timeout=10)
        r.raise_for_status()
        data = r.json()
        _CACHE[key] = data
        return data
    except Exception as e:
        print(f'  [OL API] {e}')
        return None


def ol_search(query, limit=8):
    params = {
        'q': query, 'limit': limit,
        'fields': 'key,title,author_name,first_publish_year,cover_i,isbn',
    }
    data = _ol_get(f'{OL_BASE}/search.json', params)
    if not data or 'docs' not in data:
        return []
    results = []
    for doc in data['docs']:
        authors = doc.get('author_name', ['Unknown'])
        isbn    = (doc.get('isbn') or [None])[0]
        results.append({
            'ol_key':       doc.get('key', ''),
            'title':        doc.get('title', 'Unknown'),
            'authors':      ', '.join(authors),
            'year':         doc.get('first_publish_year', 'N/A'),
            'isbn':         isbn,
            'text_feature': doc.get('title', '') + ' ' + ' '.join(authors),
            'source':       'openlibrary',
        })
    return results


print('✅ Cover helper and cold-start search ready.')

✅ Cover helper and cold-start search ready.


## 10. Meta-Model (LinearRegression Calibrator)

A `LinearRegression` is fitted on **train-set predictions** (features: collab score,
popularity, user activity). It is then evaluated on the held-out **test set** (R²)
to confirm it genuinely adds signal rather than just over-fitting.

In [12]:
train_preds = svd_model.test(trainset.build_testset())

def make_meta_features(preds):
    X, y = [], []
    for uid, iid, true_r, est, _ in preds:
        idx = int(iid)
        pop = float(books.iloc[idx]['popularity_score']) if 0 <= idx < len(books) else 0
        act = min(user_activity.get(uid, 0), 500) / 500
        X.append([est / 5, pop, act])
        y.append(true_r)
    return X, y

X_train, y_train = make_meta_features(train_preds)
X_test,  y_test  = make_meta_features(svd_predictions)

reg_model = LinearRegression().fit(X_train, y_train)

train_r2 = reg_model.score(X_train, y_train)
test_r2  = reg_model.score(X_test,  y_test)

print('✅ Meta LinearRegression trained.')
print(f'   Features     : [collab_score, popularity, user_activity]')
print(f'   Coefficients : {np.round(reg_model.coef_, 4)}')
print(f'   Intercept    : {reg_model.intercept_:.4f}')
print(f'   Train R²     : {train_r2:.4f}')
print(f'   Test  R²     : {test_r2:.4f}  ← held-out test split')

✅ Meta LinearRegression trained.
   Features     : [collab_score, popularity, user_activity]
   Coefficients : [ 8.1796 -0.1936  0.1094]
   Intercept    : -2.5063
   Train R²     : 0.6756
   Test  R²     : 0.0729  ← held-out test split


## 11. Unified Hybrid Recommendation Engine

**Pathway A — GoodBooks catalogue hit**
```
hybrid_score  = user_weight × collab + (1 − user_weight) × content
context_score = hybrid_score + 0.1 × popularity
final_score   = 0.7 × context_score + 0.3 × (reg_pred / 5)
```

**Pathway B — Cold-start** (book not in GoodBooks)
Open Library Search → encode text → FAISS → content-only scores.

In [13]:
def compute_user_weight(user_id, threshold=50):
    """Active users lean on CF; new users lean on content."""
    return min(user_activity.get(user_id, 0) / threshold, 1.0)


def _score_candidates(candidate_indices, candidate_sims, user_id):
    uw  = compute_user_weight(user_id)
    act = user_activity.get(user_id, 0)
    results = []
    for idx, cos_sim in zip(candidate_indices, candidate_sims):
        row        = books.iloc[idx]
        pop        = float(row['popularity_score'])
        collab_est = svd_model.predict(user_id, idx).est

        hybrid_score  = uw * (collab_est / 5) + (1 - uw) * float(cos_sim)
        context_score = hybrid_score + 0.1 * pop

        reg_feat   = [[collab_est / 5, pop, min(act, 500) / 500]]
        reg_pred   = float(np.clip(reg_model.predict(reg_feat)[0] / 5, 0, 1))
        final_score = 0.7 * context_score + 0.3 * reg_pred

        reasons = []
        if cos_sim > 0.6:    reasons.append('very similar content')
        elif cos_sim > 0.4:  reasons.append('similar content')
        if collab_est > 3.8: reasons.append('highly rated by similar users')
        if pop > 0.7:        reasons.append('very popular')
        explanation = ', '.join(reasons) or 'good overall match'

        results.append({
            'title':         row['title'],
            'authors':       row['authors'],
            'year':          row.get('original_publication_year', 'N/A'),
            'isbn':          str(row.get('isbn', '') or ''),
            'final_score':   round(final_score, 4),
            'content_score': round(float(cos_sim), 3),
            'collab_est':    round(collab_est, 2),
            'explanation':   explanation,
            'source':        'local',
        })
    results.sort(key=lambda x: x['final_score'], reverse=True)
    return results


title_lower_to_idx = {t.lower(): i for i, t in enumerate(books['title'])}


def smart_recommend(query_title, user_id=1, top_n=5):
    local_idx = title_lower_to_idx.get(query_title.strip().lower())
    if local_idx is not None:
        query_vec = embeddings[local_idx].reshape(1, -1)
        cands     = content_recommend_from_vec(query_vec, top_n=top_n * 4,
                                               exclude_idx=local_idx)
        seed_meta = books.iloc[local_idx].to_dict()
        seed_meta['source'] = 'local'
    else:
        ol_results = ol_search(query_title, limit=1)
        if not ol_results:
            return None, None
        seed_meta = ol_results[0]
        query_vec = encode_query(seed_meta['text_feature'])
        cands     = content_recommend_from_vec(query_vec, top_n=top_n * 4)

    idxs = [c[0] for c in cands]
    sims = [c[1] for c in cands]
    recs = _score_candidates(idxs, sims, user_id)[:top_n]
    return seed_meta, recs


# Smoke test
seed, recs = smart_recommend('The Hobbit', user_id=1)
print(f'Seed: {seed["title"] if seed else "NOT FOUND"}  '
      f'(source: {seed.get("source","?") if seed else "-"})')
if recs:
    for i, r in enumerate(recs, 1):
        print(f'  {i}. [{r["final_score"]}] {r["title"]} — {r["explanation"]}')

Seed: The Hobbit  (source: local)
  1. [0.8872] The Fellowship of the Ring (The Lord of the Rings, #1) — very similar content, highly rated by similar users
  2. [0.8783] The Lord of the Rings (The Lord of the Rings, #1-3) — very similar content, highly rated by similar users
  3. [0.8523] J.R.R. Tolkien 4-Book Boxed Set: The Hobbit and The Lord of the Rings — very similar content, highly rated by similar users
  4. [0.8481] Fool's Errand (Tawny Man, #1) — similar content, highly rated by similar users
  5. [0.8449] The Two Towers (The Lord of the Rings, #2) — very similar content, highly rated by similar users


## 12. ipywidgets Interactive UI

- **Search box** — type any title; suggestions come from GoodBooks catalogue first,
  then Open Library for cold-start books
- **Cover thumbnails** — fetched from Open Library Covers API by ISBN
- **User ID + Top-N controls**
- **Score breakdown** — content, CF estimate, final score per result
- **Source badge** — LOCAL (GoodBooks) vs OPEN LIBRARY (cold-start)

In [19]:
style = {"description_width": "90px"}

search_box = widgets.Text(
    placeholder="Type a book title",
    description="📖 Book:",
    layout=widgets.Layout(width="55%"),
    style=style,
)

suggest_dropdown = widgets.Dropdown(
    options=[],
    description="Suggestions:",
    layout=widgets.Layout(width="60%", display="none"),
    style=style,
)

user_input = widgets.BoundedIntText(
    value=1, min=1, max=53424,
    description="👤 User ID:",
    layout=widgets.Layout(width="30%"),
    style=style,
)

topn_slider = widgets.IntSlider(
    value=5, min=3, max=15, step=1,
    description="🔢 Top N:",
    layout=widgets.Layout(width="45%"),
)

search_btn    = widgets.Button(description="🔍 Search",
                               button_style="info",
                               layout=widgets.Layout(width="140px"))
recommend_btn = widgets.Button(description="📚 Recommend",
                               button_style="success",
                               layout=widgets.Layout(width="160px"))
status_label  = widgets.HTML(value="")
output_area   = widgets.Output()


def book_card_html(rec, rank):
    isbn    = rec.get("isbn", "")
    cov_url = cover_url_by_isbn(isbn, "S")
    img_tag = (
        f'<img src="{cov_url}" style="height:80px;border-radius:4px;'
        f'margin-right:12px;box-shadow:1px 1px 4px #aaa" />'
        if cov_url else
        '<div style="width:52px;height:80px;background:#e9e9e9;border-radius:4px;'
        'margin-right:12px;display:flex;align-items:center;justify-content:center;'
        'font-size:10px;color:#999">No cover</div>'
    )
    bar_w = int(rec["final_score"] * 120)
    bar   = (f'<div style="background:#4CAF50;height:6px;width:{bar_w}px;'
             f'border-radius:3px;margin-top:4px"></div>')
    return (
        '<div style="display:flex;align-items:flex-start;border:1px solid #ddd;'
        'border-radius:8px;padding:10px 14px;margin:6px 0;background:#fafafa;">'
        + img_tag +
        '<div style="flex:1">'
        f'<div style="font-size:15px;font-weight:bold">{rank}. {rec["title"]}</div>'
        f'<div style="color:#555;font-size:13px">✍️ {rec["authors"]} &nbsp;|'
        f'&nbsp; 📅 {rec["year"]}</div>'
        f'<div style="font-size:12px;color:#333;margin-top:4px">'
        f'<b>Final score:</b> {rec["final_score"]} {bar}'
        f'<span style="margin-left:12px">Content: {rec["content_score"]} '
        f'&nbsp;|&nbsp; CF est: {rec["collab_est"]}/5</span></div>'
        f'<div style="font-size:12px;color:#777;margin-top:2px">💡 {rec["explanation"]}</div>'
        '</div></div>'
    )


def on_search_click(b):
    q = search_box.value.strip().lower()
    if len(q) < 2:
        return
    local_hits = books[books["title"].str.lower().str.contains(q, na=False)].head(8)
    if not local_hits.empty:
        opts = [f'{r["title"]} — {r["authors"]}' for _, r in local_hits.iterrows()]
        suggest_dropdown.options = opts
        suggest_dropdown.layout.display = "flex"
        status_label.value = f'<span style="color:green">Found {len(opts)} matches in GoodBooks catalogue.</span>'
    else:
        ol_res = ol_search(search_box.value.strip(), limit=8)
        if ol_res:
            opts = [f'{r["title"]} — {r["authors"]}' for r in ol_res]
            suggest_dropdown.options = opts
            suggest_dropdown.layout.display = "flex"
            status_label.value = f'<span style="color:#1a73e8">No local match — showing {len(opts)} Open Library results.</span>'
        else:
            suggest_dropdown.layout.display = "none"
            status_label.value = '<span style="color:red">No results found.</span>'

search_btn.on_click(on_search_click)


def on_recommend_click(b):
    with output_area:
        clear_output(wait=True)
        if suggest_dropdown.layout.display != "none" and suggest_dropdown.value:
            title = suggest_dropdown.value.split(" — ")[0].strip()
        else:
            title = search_box.value.strip()
        if not title:
            display(HTML('<p style="color:red">Please type a book title and click Search first.</p>'))
            return
        uid   = user_input.value
        top_n = topn_slider.value
        status_label.value = f'<i style="color:#888">Generating recommendations for "{title}" …</i>'
        seed, recs = smart_recommend(title, user_id=uid, top_n=top_n)
        if seed is None:
            display(HTML(f'<p style="color:red">Could not find "{title}". Try a different title.</p>'))
            status_label.value = ""
            return
        src_color = "#2196F3" if seed.get("source") == "openlibrary" else "#4CAF50"
        src_label = "🌐 Open Library (cold-start)" if seed.get("source") == "openlibrary" else "📂 GoodBooks-10k"
        src_badge = (f'<span style="background:{src_color};color:#fff;border-radius:4px;'
                     f'padding:2px 8px;font-size:11px">{src_label}</span>')
        uw  = compute_user_weight(uid)
        act = user_activity.get(uid, 0)
        mode = "content-heavy" if uw < 0.3 else ("balanced" if uw < 0.7 else "CF-heavy")
        header = (
            f'<div style="border-left:4px solid {src_color};padding:10px 14px;'
            f'margin:8px 0;background:#f0f8ff;border-radius:0 8px 8px 0">'
            f'<div style="font-size:17px;font-weight:bold">'
            f'📖 {seed.get("title","?")} &nbsp; {src_badge}</div>'
            f'<div style="color:#555;font-size:13px">✍️ {seed.get("authors","?")} '
            f'&nbsp;|\&nbsp; 📅 {seed.get("original_publication_year", seed.get("year","?"))}</div>'
            f'<div style="font-size:12px;color:#333;margin-top:6px">'
            f'👤 User {uid} &nbsp;|\&nbsp; Ratings given: {act} &nbsp;|\&nbsp; '
            f'Adaptive weight: <b>{uw:.2f}</b> ({mode})</div></div>'
            f'<h4 style="margin:10px 0 4px">Top {top_n} Recommendations</h4>'
        )
        cards = "".join(book_card_html(r, i + 1) for i, r in enumerate(recs))
        note  = (
            '<div style="font-size:11px;color:#777;margin-top:10px;padding:8px;'
            'background:#f9f9f9;border-radius:6px"><b>Scoring formula:</b><br/>'
            'hybrid_score &nbsp;= user_weight × collab + (1−user_weight) × content<br/>'
            'context_score = hybrid_score + 0.1 × popularity<br/>'
            'final_score &nbsp;&nbsp;= 0.7 × context_score + 0.3 × (reg_pred / 5)</div>'
        )
        display(HTML(header + cards + note))
        status_label.value = '<span style="color:green">✅ Done!</span>'

recommend_btn.on_click(on_recommend_click)

banner = widgets.HTML(
    '<h2 style="margin-bottom:4px">📚 Hybrid Book Recommender — GoodBooks-10k Edition</h2>'
    '<p style="color:#555;font-size:13px">10,000 books · 6M real user ratings · '
    'Sentence Embeddings + SVD + Meta-Regression · Open Library Covers</p>'
)
display(
    banner,
    widgets.HBox([search_box, search_btn]),
    suggest_dropdown,
    widgets.HBox([user_input, topn_slider, recommend_btn],
                 layout=widgets.Layout(align_items="center")),
    status_label,
    output_area,
)

<>:126: SyntaxWarning: invalid escape sequence '\&'
<>:128: SyntaxWarning: invalid escape sequence '\&'
<>:128: SyntaxWarning: invalid escape sequence '\&'
<>:126: SyntaxWarning: invalid escape sequence '\&'
<>:128: SyntaxWarning: invalid escape sequence '\&'
<>:128: SyntaxWarning: invalid escape sequence '\&'
/tmp/ipykernel_5755/475321049.py:126: SyntaxWarning: invalid escape sequence '\&'
  f'&nbsp;|\&nbsp; 📅 {seed.get("original_publication_year", seed.get("year","?"))}</div>'
/tmp/ipykernel_5755/475321049.py:128: SyntaxWarning: invalid escape sequence '\&'
  f'👤 User {uid} &nbsp;|\&nbsp; Ratings given: {act} &nbsp;|\&nbsp; '
/tmp/ipykernel_5755/475321049.py:128: SyntaxWarning: invalid escape sequence '\&'
  f'👤 User {uid} &nbsp;|\&nbsp; Ratings given: {act} &nbsp;|\&nbsp; '


HTML(value='<h2 style="margin-bottom:4px">📚 Hybrid Book Recommender — GoodBooks-10k Edition</h2><p style="colo…

Dropdown(description='Suggestions:', layout=Layout(display='none', width='60%'), options=(), style=Description…

HTML(value='')

Output()

## 13. Programmatic Examples (No UI)

In [15]:
# Example A — in GoodBooks catalogue
seed, recs = smart_recommend('The Hunger Games', user_id=42)
if seed:
    print(f'Seed: {seed["title"]}  (source: {seed["source"]})')
    for i, r in enumerate(recs, 1):
        print(f'  {i}. [{r["final_score"]}] {r["title"]} — {r["explanation"]}')

Seed: The Hunger Games  (source: openlibrary)
  1. [0.9304] Hunger — similar content, highly rated by similar users
  2. [0.93] The Hunger Games Trilogy Boxset (The Hunger Games, #1-3) — very similar content, highly rated by similar users
  3. [0.9166] The Hunger Games (The Hunger Games, #1) — very similar content, highly rated by similar users, very popular
  4. [0.8909] Hunger Makes Me a Modern Girl — similar content, highly rated by similar users
  5. [0.831] The Hunger Games Tribute Guide — very similar content, highly rated by similar users


In [23]:
# Example B — cold-start book not in GoodBooks
seed, recs = smart_recommend('Secret of the old clock', user_id=7)
if seed:
    print(f'Seed: {seed["title"]}  (source: {seed["source"]})')
    for i, r in enumerate(recs, 1):
        print(f'  {i}. [{r["final_score"]}] {r["title"]}')
else:
    print('Not found.')

Seed: The Secret of the Old Clock  (source: openlibrary)
  1. [0.887] The Secret Keeper
  2. [0.8595] Clockwork Prince (The Infernal Devices, #2)
  3. [0.8415] Clockwork Princess (The Infernal Devices, #3)
  4. [0.8371] Secrets of a Charmed Life
  5. [0.8361] Clockwork Angel (The Infernal Devices, #1)


## 14. Model Insight Summary

In [24]:
test_user = 1
uw  = compute_user_weight(test_user)
act = user_activity.get(test_user, 0)

print('============== Model Insight ==============')
print(f'Dataset         : GoodBooks-10k (real user ratings)')
print(f'Books           : {len(books):,}')
print(f'Total ratings   : {len(ratings_df):,}')
print(f'Embedding model : all-MiniLM-L6-v2  (dim={dim})')
print(f'FAISS index     : IndexFlatIP (cosine on L2-normalised vectors)')
print(f'SVD factors     : 100  |  epochs: 20')
print()
print(f'User {test_user} profile:')
print(f'  Ratings given   : {act}')
print(f'  Adaptive weight : {uw:.2f}')
print(f'  Mode            : {"CF-heavy" if uw > 0.7 else "balanced" if uw > 0.3 else "content-heavy"}')
print()
print('Evaluation (held-out real test split):')
print(f'  RMSE         : {rmse:.4f}')
print(f'  Precision@10 : {avg_precision:.4f}')
print(f'  Recall@10    : {avg_recall:.4f}')
print(f'  NDCG@10      : {ndcg:.4f}')
print()
print('Meta-model (LinearRegression calibrator):')
print(f'  Train R²     : {train_r2:.4f}')
print(f'  Test  R²     : {test_r2:.4f}')
print('===========================================')

============== Model Insight ==============
Dataset         : GoodBooks-10k (real user ratings)
Books           : 10,000
Total ratings   : 5,976,479
Embedding model : all-MiniLM-L6-v2  (dim=384)
FAISS index     : IndexFlatIP (cosine on L2-normalised vectors)
SVD factors     : 100  |  epochs: 20

User 1 profile:
  Ratings given   : 117
  Adaptive weight : 1.00
  Mode            : CF-heavy

Evaluation (held-out real test split):
  RMSE         : 0.9160
  Precision@10 : 0.6768
  Recall@10    : 0.7586
  NDCG@10      : 0.7958

Meta-model (LinearRegression calibrator):
  Train R²     : 0.6756
  Test  R²     : 0.0729
